# Script de Segmentation des pages

* utilisation du model : pour les imprimés du 16eme  :


> Goy, F. (2026). Layout-16th-Print-Lat (v1.0.0). Zenodo. https://doi.org/10.5281/zenodo.18492102

pour des grands ensembles de données procéder avec des scripts batch, cf. documentation complète :

>

```bibtex
@misc{Goy_Documentations,
  author={Floriane Goy},
  title={Documentations on digital frameworks for Pauline exegesis project},
  version={1.0},
  address={Genève},
  publisher={université de Genève},
  year={2023-2026},
  url={https://github.com/16thExegesisDH/Documentations},
}
```

## Setup

installer les bibliothèques python

In [1]:
!python --version

Python 3.12.13


Installer yaltai, cela peut prendre du temps.

In [2]:
!pip install yaltai

lier le notebook a googlecolab

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


mettre en place les dossiers

In [3]:
!mkdir content
!mkdir content/image

mkdir: cannot create directory ‘content’: File exists
mkdir: cannot create directory ‘content/image’: File exists
mkdir: cannot create directory ‘model/’: File exists


Télécharger le modèle de segmentation

In [4]:
!wget https://github.com/16thExegesisDH/Segmentation_model/releases/download/v1.0.0/Layout-16th-Print-Lat.pt

--2026-05-07 08:04:14--  https://github.com/16thExegesisDH/Segmentation_model/releases/download/v1.0.0/Layout-16th-Print-Lat.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1057247382/f505ab75-d207-4e8d-8302-fcb9b297eb5d?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-07T08%3A38%3A02Z&rscd=attachment%3B+filename%3DLayout-16th-Print-Lat.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-07T07%3A37%3A23Z&ske=2026-05-07T08%3A38%3A02Z&sks=b&skv=2018-11-09&sig=evsBfcLqCT20Eb%2FQ7sjnBjZXBsit94okNCSdq1xwlPE%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3ODE0Mjg1NCwibmJmIjoxNzc4MTQxMDU0LCJwYXR

mettre dans image le fichier.zip contenant les images à traiter.

dezipper le fichier

In [15]:
!unzip content/image/10AC1F79_bugenhagen.zip -d content/image/

Archive:  content/image/10AC1F79_bugenhagen.zip
  inflating: content/image/10AC1F79_uk4nGb4s1jU1XvfK_00325.jpg  
  inflating: content/image/10AC1F79_uk4nGb4s2jTSka17_00347.jpg  
  inflating: content/image/10AC1F79_uk4nGb4s3EZhtipH_00359.jpg  
  inflating: content/image/10AC1F79_uk4nGb4s3itAK4dX_00364.jpg  


supprimer le fichier zip

In [16]:
!rm -rf content/image/10AC1F79_bugenhagen.zip

## Segmenter

 lancer le script

*NB* : sans l'utilisation de cpu le traitement des image est assez long pour le traitement des grands ensembles de donner utiliser le cluster HPC.

environ 1 minutes par images sur collab


In [5]:
#yaltai kraken --verbose  -I "content/image/*.jpg" --alto --suffix ".xml" segment --yolo Layout-16th-Print-Lat.pt
!yaltai kraken --verbose  -I "content/image/*.jpg" --alto --suffix ".xml" segment --yolo /content/Layout-16th-Print-Lat.pt


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

image 1/1 /content/content/image/10AC1F79_uk4nGb4s3EZhtipH_00359.jpg: 960x640 1 NumberingZone, 1 RunningTitleZone, 7 MainZone-Heads, 5 MainZone-Ps, 1 QuireMarksZone, 2621.9ms
Speed: 13.8ms preprocess, 2621.9ms inference, 1.5ms postprocess per image at shape (1, 3, 960, 640)
[05/07/26 08:08:18] INFO     Vectorizing baselines                   ]8;id=919216;file:///usr/local/lib/python3.12/dist-packages/kraken/blla.py\blla.py]8;;\:]8;id=603318;file:///usr/local/lib/python3.12/dist-packages/kraken/blla.py#211\211]8;;\
INFO:kraken.blla:Vectorizing baselines
[05/07/26 08:08:33] INFO     Compute reading order on 2 lines ]8;id=725173;file:///usr/local/lib/python3.12/dist-package

traitement des données post segmentation

corrige les fins de fichiers, si la segmentation a bugué  (normalement ça n'arrive pas quand on traite des petits jeux de données

In [6]:
"""
The end of the ALTO file can be corrupted du to the segmentation work. This script correct the eventual bug appearing on the last line of the alto file.
"""
import os

FOLDER_PATH = "content/image/"  # change this

for file_end in os.listdir(FOLDER_PATH):
    if file_end.lower().endswith(".xml"):
        file_path = os.path.join(FOLDER_PATH, file_end)

        with open(file_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        if not lines:
            continue  # skip empty files

        # Remove the last line
        lines = lines[:-1]

        # Add the correct closing tag
        lines.append("</alto>\n")

        # Rewrite the file
        with open(file_path, "w", encoding="utf-8") as f:
            f.writelines(lines)

        print(f"Fixed: {file_end}")


Fixed: 10AC1F79_uk4nGb4s2jTSka17_00347.xml
Fixed: 10AC1F79_uk4nGb4s3EZhtipH_00359.xml
Fixed: 10AC1F79_uk4nGb4s3itAK4dX_00364.xml
Fixed: 10AC1F79_uk4nGb4s1jU1XvfK_00325.xml


homogéniser le nom des fichier `jpg` et leur identifiant `xml`

In [7]:
"""
The script give your xml_file a name corresponding to your image_file so you can use the both files in eScriptorium
"""
import os
import fileinput

for file in os.listdir(os.path.join("content","image")):
    if file.endswith(".xml"):
      with fileinput.FileInput(os.path.join("content","image",file), inplace=True) as f:
        for line in f:
          #modify the path according to the <FileName> of your ALTO file
          print(line.replace('content/image/',''), end='')
print("All files are corrected!")



All files are corrected!


nommer toutes les lignes : `Defaultline`

In [8]:
import glob
import re

folder_path = "content/image/*.xml"
# we are using regex so it works even on corrupted xml
pattern = re.compile(
    r'(<OtherTag\s+[^>]*DESCRIPTION="line type"[^>]*ID="LINE_TYPE_1"[^>]*TYPE="type"[^>]*LABEL=")default(")',
    re.IGNORECASE
)

for file_path in glob.glob(folder_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        new_content, count = pattern.subn(r'\1DefaultLine\2', content)

        if count > 0:
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(new_content)
            print(f"Updated {count} tag(s) in: {file_path}")

    except Exception as e:
        print(f"Skipped {file_path}: {e}")


Updated 1 tag(s) in: content/image/10AC1F79_uk4nGb4s2jTSka17_00347.xml
Updated 1 tag(s) in: content/image/10AC1F79_uk4nGb4s3EZhtipH_00359.xml
Updated 1 tag(s) in: content/image/10AC1F79_uk4nGb4s3itAK4dX_00364.xml
Updated 1 tag(s) in: content/image/10AC1F79_uk4nGb4s1jU1XvfK_00325.xml


mets en forme les identifiants alphanumérique  du model de segmentation des éléments   `Block` et  `line`.  

In [9]:
import glob
from lxml import etree

folder_path = "content/image/*.xml"

ALTO_NS = "http://www.loc.gov/standards/alto/ns-v4#"
NSMAP = {"alto": ALTO_NS}

# Preserve formatting when writing
parser = etree.XMLParser(remove_blank_text=False)

for file_path in glob.glob(folder_path):
    try:
        tree = etree.parse(file_path, parser)
        root = tree.getroot()

        block_counter = 0
        line_counter = 0

        # Renumber TextBlock
        for block in root.xpath(".//alto:TextBlock", namespaces=NSMAP):
            block_counter += 1
            block.set("ID", f"block_{block_counter}")

        # Renumber TextLine
        for line in root.xpath(".//alto:TextLine", namespaces=NSMAP):
            line_counter += 1
            line.set("ID", f"line_{line_counter}")

        # Write back with pretty formatting preserved
        tree.write(
            file_path,
            encoding="utf-8",
            xml_declaration=True,
            pretty_print=True
        )

        print(
            f"Updated {block_counter} TextBlock(s) and "
            f"{line_counter} TextLine(s) in: {file_path}"
        )

    except Exception as e:
        print(f"Skipped {file_path}: {e}")

Updated 11 TextBlock(s) and 28 TextLine(s) in: content/image/10AC1F79_uk4nGb4s2jTSka17_00347.xml
Updated 15 TextBlock(s) and 22 TextLine(s) in: content/image/10AC1F79_uk4nGb4s3EZhtipH_00359.xml
Updated 7 TextBlock(s) and 31 TextLine(s) in: content/image/10AC1F79_uk4nGb4s3itAK4dX_00364.xml
Updated 16 TextBlock(s) and 26 TextLine(s) in: content/image/10AC1F79_uk4nGb4s1jU1XvfK_00325.xml


compile les fichier xml (contenant la segmentation) en un fichier `.zip

In [10]:
!zip -r altos_segmented.zip content/image/*xml

  adding: content/image/10AC1F79_uk4nGb4s1jU1XvfK_00325.xml (deflated 74%)
  adding: content/image/10AC1F79_uk4nGb4s2jTSka17_00347.xml (deflated 73%)
  adding: content/image/10AC1F79_uk4nGb4s3EZhtipH_00359.xml (deflated 75%)
  adding: content/image/10AC1F79_uk4nGb4s3itAK4dX_00364.xml (deflated 74%)


télécharge le fichier zip

In [11]:

from google.colab import files
files.download('/content/altos_segmented.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

supprimer les fichiers de image

In [12]:
! rm content/image/*

Recommencer avec de nouvelles images en fichier zip.